# Cleaning

<span style="color: #777777; font-style: italic;">정제</span>

*Fix known format differences and handle missing values and repeated records.*  
<span style="color: #777777; font-style: italic;">정제(cleaning)로 표기를 통일하고 결측값과 중복 기록을 처리합니다.</span>


- Clean the text while keeping a missing score separate from zero.  
  <span style="color: #777777; font-style: italic;">텍스트를 정리하면서 결측 점수와 0점을 구분합니다.</span>

  <img src="https://raw.githubusercontent.com/sonamu-jun/introduction-to-bigdata/main/05-2_Data_Wrangling/assets/03_cleaning/concept.webp" width="524" alt="Clean the text while keeping a missing score separate from zero.">


## Spaces and Letter Case

<span style="color: #777777; font-style: italic;">공백과 대소문자</span>

- Use `.copy()` to keep the original table.  
  <span style="color: #777777; font-style: italic;">`.copy()`로 복사해 원본 표를 보관합니다.</span>

- `.str.strip()` removes spaces at the start and end.  
  <span style="color: #777777; font-style: italic;">`.str.strip()`은 문자열 앞뒤의 공백을 지웁니다.</span>

- `.str.title()` starts each word with a capital; `.str.upper()` makes all letters capitals.  
  <span style="color: #777777; font-style: italic;">`.str.title()`은 단어 첫 글자를, `.str.upper()`는 모든 글자를 대문자로 바꿉니다.</span>


In [1]:
import pandas as pd

raw_df = pd.DataFrame(
    {
        "name": ["  jiyoung ", " HOJUN", "hana "],
        "subject": [" mathematics", "ENGLISH ", " science"],
        "city": ["seoul ", "  Busan", "  daegu "],
    }
)
clean_df = raw_df.copy()
clean_df["name"] = clean_df["name"].str.strip().str.title()
clean_df["subject"] = clean_df["subject"].str.strip().str.title()
clean_df["city"] = clean_df["city"].str.strip().str.upper()

print(clean_df)
print("Original first name:", repr(raw_df.loc[0, "name"]))


      name      subject   city
0  Jiyoung  Mathematics  SEOUL
1    Hojun      English  BUSAN
2     Hana      Science  DAEGU
Original first name: '  jiyoung '


- The names become Jiyoung, Hojun, and Hana; city names use capitals.  
  <span style="color: #777777; font-style: italic;">이름은 Jiyoung, Hojun, Hana로 바뀌고 도시 이름은 대문자로 통일됩니다.</span>

- The raw names stay unchanged; real names may need different case rules.  
  <span style="color: #777777; font-style: italic;">원본 이름은 그대로 남으며, 실제 이름에는 다른 대소문자 규칙이 필요할 수 있습니다.</span>


## Category Mapping

<span style="color: #777777; font-style: italic;">범주 매핑</span>

- `Category mapping` uses a dictionary to give labels with the same meaning one name.  
  <span style="color: #777777; font-style: italic;">`범주 매핑`(category mapping)은 딕셔너리로 같은 뜻의 표기를 하나로 통일합니다.</span>

- `.str.lower()` makes matching easier; `.map()` applies the dictionary.  
  <span style="color: #777777; font-style: italic;">`.str.lower()`로 소문자를 통일하고 `.map()`으로 대응표를 적용합니다.</span>


In [2]:
subject_labels = pd.Series(["math", "Mathematics ", " ENG", "english", "biology"])
subject_mapping = {
    "math": "Mathematics",
    "mathematics": "Mathematics",
    "eng": "English",
    "english": "English",
}

subject_keys = subject_labels.str.strip().str.lower()
mapped_subjects = subject_keys.map(subject_mapping)

print(mapped_subjects)


0    Mathematics
1    Mathematics
2        English
3        English
4            NaN
dtype: str


- Four labels become Mathematics or English.  
  <span style="color: #777777; font-style: italic;">네 표기가 Mathematics 또는 English로 통일됩니다.</span>

- Biology is not in the dictionary, so the result is `NaN`, a missing-value marker here.  
  <span style="color: #777777; font-style: italic;">Biology는 대응표에 없어 `NaN`으로 표시되며, 여기서 `NaN`은 결측값을 나타냅니다.</span>


## Missing Values and Errors

<span style="color: #777777; font-style: italic;">결측값과 오류</span>

- Here `None`, `NA`, empty text, and the text `None` all mean a missing score.  
  <span style="color: #777777; font-style: italic;">여기서는 `None`, `NA`, 빈 문자열, 문자열 `None`이 모두 점수 누락을 뜻합니다.</span>

- `.replace()` makes the missing labels consistent; `pd.to_numeric(..., errors="coerce")` turns invalid text into `NaN`.  
  <span style="color: #777777; font-style: italic;">`.replace()`로 누락 표시를 통일하고, `pd.to_numeric(..., errors="coerce")`로 숫자가 아닌 텍스트를 `NaN`으로 바꿉니다.</span>

- `.notna()` marks present inputs; if conversion makes one missing, it is a format error.  
  <span style="color: #777777; font-style: italic;">`.notna()`는 값이 있는 입력을 표시하며, 그런 값이 변환 후 결측값이 되면 형식 오류입니다.</span>

- A `Series` is a labeled, one-dimensional set of values; use `&` for both conditions and `|` for either or both.  
  <span style="color: #777777; font-style: italic;">`시리즈`(Series)는 인덱스가 붙은 1차원 자료이며, `&`는 두 조건 모두, `|`는 하나 이상을 만족한다는 뜻입니다.</span>


In [3]:
raw_scores = pd.Series(["100", None, "NA", "", "None", "eighty", "105", "-3", "86"])

tokens = raw_scores.replace({"NA": None, "": None, "None": None})
numeric_scores = pd.to_numeric(tokens, errors="coerce")
missing_input = tokens.isna()
format_error = tokens.notna() & numeric_scores.isna()
range_error = (numeric_scores < 0) | (numeric_scores > 100)

print("Missing:", missing_input.sum())
print("Format errors:", format_error.sum())
print("Range errors:", range_error.sum())


Missing: 4
Format errors: 1
Range errors: 2


- There are four missing scores, one format error (`eighty`), and two range errors (105 and -3).  
  <span style="color: #777777; font-style: italic;">점수 누락 4개, 형식 오류 1개(`eighty`), 범위 오류 2개(105와 -3)가 있습니다.</span>


## Keeping Valid Scores

<span style="color: #777777; font-style: italic;">유효 점수 선택</span>

- `.mask(range_error)` marks out-of-range scores as missing; the raw scores and error labels stay available.  
  <span style="color: #777777; font-style: italic;">`.mask(range_error)`는 범위를 벗어난 점수를 결측값으로 바꾸며, 원본 점수와 오류 표시는 남깁니다.</span>

- `.mean()` skips missing values; `.fillna(0)` replaces them with zero.  
  <span style="color: #777777; font-style: italic;">`.mean()`은 결측값을 빼고 계산하며, `.fillna(0)`은 결측값을 0으로 채웁니다.</span>


In [4]:
clean_scores = numeric_scores.mask(range_error)
filled_scores = clean_scores.fillna(0)

print(clean_scores)
print("Count:", clean_scores.count())
print("Mean:", clean_scores.mean())
print("Mean with zeros:", round(filled_scores.mean(), 2))


0    100.0
1      NaN
2      NaN
3      NaN
4      NaN
5      NaN
6      NaN
7      NaN
8     86.0
dtype: float64
Count: 2
Mean: 93.0
Mean with zeros: 20.67


- The two valid scores, 100 and 86, have mean 93.  
  <span style="color: #777777; font-style: italic;">유효한 두 점수 100과 86의 평균은 93입니다.</span>

- Filling the other seven scores with zero lowers the mean to 20.67.  
  <span style="color: #777777; font-style: italic;">나머지 점수 7개를 0으로 채우면 평균이 20.67로 낮아집니다.</span>

- A missing score does not mean the student scored zero.  
  <span style="color: #777777; font-style: italic;">점수가 빠졌다고 실제 0점을 받은 것은 아닙니다.</span>


## Date Formats

<span style="color: #777777; font-style: italic;">날짜 형식</span>

- `strptime()` reads a date using a given format: `%Y` is year, `%m` is month, and `%d` is day.  
  <span style="color: #777777; font-style: italic;">`strptime()`은 지정한 형식으로 날짜를 읽으며, `%Y`는 연도, `%m`은 월, `%d`는 일입니다.</span>

- The source defines `01-09-2025` as day–month–year; `isoformat()` writes year–month–day.  
  <span style="color: #777777; font-style: italic;">입력에서 `01-09-2025`는 일–월–년 순서이며, `isoformat()`은 년–월–일로 출력합니다.</span>


In [5]:
from datetime import datetime

dates = [
    ("2025/09/01", "%Y/%m/%d"),
    ("01-09-2025", "%d-%m-%Y"),
    ("2025.9.1", "%Y.%m.%d"),
]

for text, date_format in dates:
    date = datetime.strptime(text, date_format).date()
    print(date.isoformat())

try:
    datetime.strptime("2025/02/30", "%Y/%m/%d")
except ValueError:
    print("Invalid date: 2025/02/30")


2025-09-01
2025-09-01
2025-09-01
Invalid date: 2025/02/30


- All three valid dates become `2025-09-01`.  
  <span style="color: #777777; font-style: italic;">유효한 날짜 3개는 모두 `2025-09-01`로 바뀝니다.</span>

- February 30 does not exist; `except ValueError` handles this error.  
  <span style="color: #777777; font-style: italic;">2월 30일은 없는 날짜이며, `except ValueError`로 이 오류를 처리합니다.</span>


## Repeated Records

<span style="color: #777777; font-style: italic;">반복된 기록</span>

- A `key` identifies one score entry; here the name–subject pair is a `composite key`.  
  <span style="color: #777777; font-style: italic;">`키`(key)는 점수 기록을 구분하는 기준이며, 이름–과목 쌍은 여러 항목을 묶은 `복합 키`(composite key)입니다.</span>

- Rows with the same key may be updates; a larger `update_order` means a later update here.  
  <span style="color: #777777; font-style: italic;">같은 키의 행은 갱신 기록일 수 있으며, 여기서는 `update_order`가 클수록 나중 기록입니다.</span>

- Sort by update order, then use `drop_duplicates(..., keep="last")` to keep the latest row for each key.  
  <span style="color: #777777; font-style: italic;">갱신 순서로 정렬한 뒤 `drop_duplicates(..., keep="last")`로 키마다 마지막 행을 남깁니다.</span>


In [6]:
updates_df = pd.DataFrame(
    {
        "name": ["Jiyoung", "Hojun", "Jiyoung", "Hojun"],
        "subject": ["Mathematics", "English", "Mathematics", "English"],
        "score": [95, 80, 100, 90],
        "update_order": [1, 2, 3, 4],
    }
)
key_columns = ["name", "subject"]
latest_df = updates_df.sort_values("update_order").drop_duplicates(
    key_columns, keep="last"
)
print("Duplicate rows:", updates_df.duplicated().sum())
print("Repeated keys:", updates_df.duplicated(key_columns).sum())
print(latest_df)


Duplicate rows: 0
Repeated keys: 2
      name      subject  score  update_order
2  Jiyoung  Mathematics    100             3
3    Hojun      English     90             4


- There are no `exact duplicate` rows but two repeated `keys`.  
  <span style="color: #777777; font-style: italic;">완전히 같은 행은 없지만 반복된 `키`는 2개입니다.</span>

- Retaining the latest versions selects Jiyoung's 100 and Hojun's 90.  
  <span style="color: #777777; font-style: italic;">최신 버전을 남기면 Jiyoung의 100점과 Hojun의 90점이 선택됩니다.</span>


- Keep raw values and use clear rules for each change.  
  <span style="color: #777777; font-style: italic;">원본을 보관하고 수정 기준을 명확히 합니다.</span>

- Missing scores are not zeros.  
  <span style="color: #777777; font-style: italic;">빠진 점수는 0점이 아닙니다.</span>
